In [11]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
# from IPython.display import Image
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from dotenv import load_dotenv; load_dotenv() 
import os

In [12]:
llm = HuggingFaceEndpoint(
    repo_id='deepseek-ai/DeepSeek-R1',
    task='text-generation',
    huggingfacehub_api_token=os.getenv("HF_TOKEN")
)
model = ChatHuggingFace(llm=llm)

In [13]:
class LLMState(TypedDict):

    question: str
    answer: str

In [14]:
graph = StateGraph(LLMState)

In [15]:
def question_llm(state: LLMState) -> LLMState:

    question = state['question']
    prompt = f"Answer the following question {question}"
    state['answer'] = model.invoke(prompt).content

    return state

graph.add_node('question_llm', question_llm)

In [16]:
graph.add_edge(START, 'question_llm')
graph.add_edge('question_llm', END)

In [17]:
workflow = graph.compile()

In [18]:
initial_state = {"question": "How does Google pregel works?"}
workflow.invoke(initial_state)

{'question': 'How does Google pregel works?',
 'answer': "\nGoogle Pregel is a distributed computing framework designed for **large-scale graph processing**, inspired by the **Bulk Synchronous Parallel (BSP)** model. Here's a concise breakdown of how it works:\n\n### Core Concepts\n1. **Vertex-Centric Model**:\n   - Computation revolves around **vertices** (nodes) of a graph. Each vertex:\n     - Has a unique ID and a modifiable value (state).\n     - Contains outgoing edges (with optional values).\n     - Runs a user-defined `Compute()` function.\n\n2. **Supersteps (Synchronization Barriers)**:\n   - Processing occurs in discrete iterations called **supersteps**.\n   - In each superstep:\n     - Vertices execute `Compute()` in parallel.\n     - Messages sent in superstep \\(N\\) are delivered at superstep \\(N+1\\).\n   - Global synchronization happens between supersteps.\n\n3. **Message Passing**:\n   - Vertices communicate via **messages** (e.g., vertex A sends data to vertex B).\n 